# CH02 -- Notebook thực hành
Chạy lần lượt từng ô từ trên xuống. Đặt notebook ở thư mục gốc dự án (cùng cấp với thư mục `data/` và `figures/`).

### Ô lệnh 1 -- Nạp thư viện và dữ liệu đã chia ở Chương 1

In [ ]:
import sys
sys.path.append("code")              # để Python tìm thấy file code/common.py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from common import credit_split, xgb_clf

pd.set_option("display.width", 80)
pd.set_option("display.precision", 3)
SEED = 42

X_train, X_test, y_train, y_test = credit_split()     # đúng cách chia của Bước 13, Chương 1
print("Train:", X_train.shape, "  Test:", X_test.shape)
print("Số khách hàng trong tập train theo nhãn:", y_train.value_counts().to_dict())


### Ô lệnh 2 -- Nhắc lại: XGBoost ở Chương 1 bỏ sót bao nhiêu người vỡ nợ?

In [ ]:
from sklearn.metrics import (confusion_matrix, precision_score, recall_score,
                             f1_score, average_precision_score)

xgb = xgb_clf().fit(X_train, y_train)
proba_base = xgb.predict_proba(X_test)[:, 1]
pred_base = (proba_base >= 0.5).astype(int)

TN, FP, FN, TP = confusion_matrix(y_test, pred_base).ravel()
print(f"Thực sự vỡ nợ: {TP + FN} người -> phát hiện được {TP}, bỏ sót {FN}")
print(f"Precision = {precision_score(y_test, pred_base):.3f}, "
      f"Recall = {recall_score(y_test, pred_base):.3f}, F1 = {f1_score(y_test, pred_base):.3f}")


### Ô lệnh 3 -- Một bộ dữ liệu "đồ chơi" 2 chiều để nhìn thấy SMOTE bằng mắt

In [ ]:
from sklearn.datasets import make_classification

X_toy, y_toy = make_classification(n_samples=600, n_features=2, n_redundant=0,
                                   n_clusters_per_class=1, weights=[0.9, 0.1],
                                   class_sep=1.0, random_state=3)
print("Số điểm mỗi lớp [lớp 0, lớp 1]:", np.bincount(y_toy))
print("5 điểm đầu tiên:")
print(np.round(X_toy[:5], 3))


### Ô lệnh 4 -- Cách đơn giản nhất: nhân bản ngẫu nhiên các điểm thiểu số

In [ ]:
from imblearn.over_sampling import RandomOverSampler

X_ros, y_ros = RandomOverSampler(random_state=SEED).fit_resample(X_toy, y_toy)
print("Sau khi nhân bản [lớp 0, lớp 1]:", np.bincount(y_ros))

minority = X_ros[y_ros == 1]
print("Số điểm lớp 1:", len(minority),
      "| trong đó số điểm KHÁC NHAU:", len(np.unique(minority, axis=0)))


### Ô lệnh 5 -- Bước 1 của SMOTE: tìm 5 láng giềng gần nhất (cùng lớp thiểu số)

In [ ]:
from sklearn.neighbors import NearestNeighbors

X_min = X_toy[y_toy == 1]                         # 66 điểm thiểu số
nn = NearestNeighbors(n_neighbors=6).fit(X_min)   # 6 = chính nó + 5 láng giềng
dist, idx = nn.kneighbors(X_min[[0]])             # hỏi láng giềng của điểm số 0

print("Điểm x_0          :", X_min[0].round(3))
print("Chỉ số 5 láng giềng:", idx[0][1:])           # bỏ phần tử đầu (chính nó)
print("Khoảng cách        :", dist[0][1:].round(3))


### Ô lệnh 6 -- Bước 2 của SMOTE: sinh MỘT điểm mới bằng tay

In [ ]:
rng = np.random.default_rng(SEED)
j = rng.choice(idx[0][1:])           # chọn ngẫu nhiên 1 trong 5 láng giềng
lam = rng.uniform(0, 1)              # hệ số lambda ngẫu nhiên trong khoảng (0, 1)

x_i, x_nb = X_min[0], X_min[j]
x_new = x_i + lam * (x_nb - x_i)     # công thức SMOTE

print(f"Láng giềng được chọn: điểm số {j}, lambda = {lam:.3f}")
print("x_i   =", x_i.round(3))
print("x_nb  =", x_nb.round(3))
print("x_new =", x_new.round(3))


### Ô lệnh 7 -- Tự viết hàm SMOTE hoàn chỉnh (khoảng 10 dòng)

In [ ]:
def smote_tay(X_min, G, k=5, seed=SEED):
    """Sinh G điểm tổng hợp từ tập thiểu số X_min (phiên bản tối giản để học)."""
    rng = np.random.default_rng(seed)
    _, idx = NearestNeighbors(n_neighbors=k + 1).fit(X_min).kneighbors(X_min)
    new = []
    for _ in range(G):
        i = rng.integers(len(X_min))          # chọn ngẫu nhiên một điểm thiểu số
        j = rng.choice(idx[i][1:])            # chọn một trong k láng giềng của nó
        lam = rng.uniform(0, 1)
        new.append(X_min[i] + lam * (X_min[j] - X_min[i]))
    return np.array(new)

G = (y_toy == 0).sum() - (y_toy == 1).sum()   # số điểm cần sinh để hai lớp bằng nhau
X_syn = smote_tay(X_min, G)
print("Cần sinh:", G, "điểm | Kích thước kết quả:", X_syn.shape)


### Ô lệnh 8 -- Dùng SMOTE có sẵn trong imbalanced-learn và vẽ kết quả

In [ ]:
from imblearn.over_sampling import SMOTE

X_sm, y_sm = SMOTE(k_neighbors=5, random_state=SEED).fit_resample(X_toy, y_toy)
print("Sau SMOTE [lớp 0, lớp 1]:", np.bincount(y_sm))
syn = X_sm[len(X_toy):]                          # imblearn đặt các điểm mới ở cuối mảng

fig, axes = plt.subplots(1, 2, figsize=(8, 3.3), sharex=True, sharey=True)
for ax, title in zip(axes, ["Dữ liệu gốc", "Sau SMOTE (k = 5)"]):
    ax.scatter(*X_toy[y_toy == 0].T, s=8, color="#c9c8c2", label="Lớp đa số")
    ax.scatter(*X_toy[y_toy == 1].T, s=14, color="#eb6834", label="Lớp thiểu số (gốc)")
    ax.set_title(title)
axes[1].scatter(*syn.T, s=10, marker="x", color="#2a78d6", label="Mẫu tổng hợp")
axes[1].legend(loc="lower right", fontsize=7)
plt.savefig("figures/ch02_s08_toy.pdf", bbox_inches="tight")


### Ô lệnh 9 -- Vì sao phải chuẩn hóa trước SMOTE? So sánh "láng giềng gần nhất"

In [ ]:
from sklearn.preprocessing import StandardScaler

X_min_c = X_train[y_train == 1]                     # 5 309 khách hàng vỡ nợ (tập train)
scaler = StandardScaler().fit(X_train)
nb_raw = NearestNeighbors(n_neighbors=2).fit(X_min_c).kneighbors(X_min_c)[1][:, 1]
Z = scaler.transform(X_min_c)
nb_std = NearestNeighbors(n_neighbors=2).fit(Z).kneighbors(Z)[1][:, 1]

i = 2                                               # một ví dụ cụ thể
cols = ["LIMIT_BAL", "AGE", "PAY_1", "PAY_2", "BILL_AMT1", "PAY_AMT1"]
print(pd.DataFrame({"Khách hàng": X_min_c.iloc[i][cols],
                    "LG chưa chuẩn hóa": X_min_c.iloc[nb_raw[i]][cols],
                    "LG đã chuẩn hóa": X_min_c.iloc[nb_std[i]][cols]}))

same = lambda nb: (X_min_c["PAY_1"].values == X_min_c["PAY_1"].values[nb]).mean()
print(f"\nTỷ lệ láng giềng có CÙNG PAY_1 -- chưa chuẩn hóa: {same(nb_raw):.1%},"
      f" đã chuẩn hóa: {same(nb_std):.1%}")


### Ô lệnh 10 -- SMOTE thay đổi tập train như thế nào?

In [ ]:
X_res, y_res = SMOTE(random_state=SEED).fit_resample(scaler.transform(X_train), y_train)

print("Trước SMOTE:", y_train.value_counts().to_dict())
print("Sau SMOTE  :", pd.Series(y_res).value_counts().to_dict())
print("Số mẫu tổng hợp được thêm vào:", len(y_res) - len(y_train))
print("Tập test vẫn giữ nguyên       :", y_test.value_counts().to_dict())


### Ô lệnh 11 -- So sánh có/không SMOTE bằng đánh giá chéo 5-fold

In [ ]:
from imblearn.pipeline import Pipeline          # CHÚ Ý: Pipeline của imblearn, không phải sklearn
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {"ROC-AUC": "roc_auc", "PR-AUC": "average_precision",
           "F1": "f1", "Recall": "recall", "Precision": "precision"}

def make(model, smote):
    """Ghép chuẩn hóa -> (SMOTE) -> mô hình thành một pipeline."""
    steps = [("scale", StandardScaler())]
    if smote:
        steps.append(("smote", SMOTE(random_state=SEED)))
    return Pipeline(steps + [("model", model)])

configs = {"LogReg": make(LogisticRegression(max_iter=2000), False),
           "LogReg + SMOTE": make(LogisticRegression(max_iter=2000), True),
           "XGBoost": make(xgb_clf(), False),
           "XGBoost + SMOTE": make(xgb_clf(), True)}
cv_table = {}
for name, pipe in configs.items():
    r = cross_validate(pipe, X_train, y_train, cv=skf, scoring=scoring)
    cv_table[name] = {k: r["test_" + k].mean() for k in scoring}
print(pd.DataFrame(cv_table).T)


### Ô lệnh 12 -- Kiểm tra trên tập test (ngưỡng 0,5)

In [ ]:
test_table = {}
for name, pipe in configs.items():
    pipe.fit(X_train, y_train)
    p = pipe.predict_proba(X_test)[:, 1]
    pred = (p >= 0.5).astype(int)
    test_table[name] = {"PR-AUC": average_precision_score(y_test, p),
                        "F1": f1_score(y_test, pred),
                        "Recall": recall_score(y_test, pred),
                        "Precision": precision_score(y_test, pred),
                        "Bị gắn cờ": pred.sum()}
test_df = pd.DataFrame(test_table).T
test_df["Bị gắn cờ"] = test_df["Bị gắn cờ"].astype(int)
print(test_df)


### Ô lệnh 13 -- Làm SAI: chạy SMOTE trên toàn bộ tập train RỒI MỚI chia fold

In [ ]:
X_leak, y_leak = SMOTE(random_state=SEED).fit_resample(scaler.transform(X_train), y_train)
r_wrong = cross_validate(xgb_clf(), X_leak, y_leak, cv=skf, scoring=scoring)
wrong = {k: r_wrong["test_" + k].mean() for k in scoring}

print(pd.DataFrame({"ĐÚNG (SMOTE trong pipeline)": cv_table["XGBoost + SMOTE"],
                    "SAI (SMOTE trước khi chia)": wrong}).round(3))


### Ô lệnh 14 -- Vì sao làm sai lại cho điểm cao như vậy? Đo khoảng cách

In [ ]:
real_min = Z                             # 5 309 khách hàng vỡ nợ thật (đã chuẩn hóa)
synth = X_leak[len(X_train):]                 # 13 382 điểm tổng hợp (nằm ở cuối mảng)
nn_real = NearestNeighbors(n_neighbors=2).fit(real_min)

d_real = nn_real.kneighbors(real_min)[0][:, 1]   # điểm thật -> điểm thật KHÁC gần nhất
d_syn = nn_real.kneighbors(synth)[0][:, 0]       # điểm tổng hợp -> điểm thật gần nhất
print(f"Trung vị khoảng cách: điểm thật     -> điểm thật khác gần nhất: {np.median(d_real):.3f}")
print(f"Trung vị khoảng cách: điểm tổng hợp -> điểm thật gần nhất     : {np.median(d_syn):.3f}")


### Ô lệnh 15 -- Ảnh hưởng của số láng giềng k (đánh giá chéo, mất khoảng 1 phút)

In [ ]:
rows = []
for k in [1, 3, 5, 10, 20, 50]:
    pipe = Pipeline([("scale", StandardScaler()),
                     ("smote", SMOTE(k_neighbors=k, random_state=SEED)),
                     ("model", xgb_clf())])
    r = cross_validate(pipe, X_train, y_train, cv=skf, scoring=scoring)
    rows.append({"k": k, **{m: r["test_" + m].mean()
                            for m in ["PR-AUC", "F1", "Recall", "Precision"]}})
k_table = pd.DataFrame(rows).set_index("k")
print(k_table)

ax = k_table[["PR-AUC", "F1", "Recall", "Precision"]].plot(marker="o", figsize=(5, 2.8), logx=True)
ax.set_xticks(k_table.index); ax.set_xticklabels(k_table.index)
ax.set_ylabel("Điểm CV trung bình")
plt.savefig("figures/ch02_s15_k.pdf", bbox_inches="tight")


### Ô lệnh 16 -- Một đối thủ đơn giản: giữ nguyên mô hình, chỉ dời ngưỡng

In [ ]:
from sklearn.model_selection import cross_val_predict

# Dự đoán "out-of-fold": mỗi khách hàng train được dự đoán bởi mô hình CHƯA từng thấy họ
oof = cross_val_predict(xgb_clf(), X_train, y_train, cv=skf, method="predict_proba")[:, 1]
thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)
f1s = [f1_score(y_train, (oof >= t).astype(int)) for t in thresholds]
best_t = thresholds[np.argmax(f1s)]
print(f"Ngưỡng cho F1 cao nhất (tính trên tập train): t* = {best_t:.2f}, F1 = {max(f1s):.3f}")

fig, ax = plt.subplots(figsize=(5, 2.6))
ax.plot(thresholds, f1s)
ax.axvline(best_t, ls="--", color="gray"); ax.axvline(0.5, ls=":", color="gray")
ax.set_xlabel("Ngưỡng t"); ax.set_ylabel("F1 (out-of-fold)")
plt.savefig("figures/ch02_s16_threshold.pdf", bbox_inches="tight")


### Ô lệnh 17 -- So sánh trên tập test: SMOTE hay dời ngưỡng?

In [ ]:
p_smote = configs["XGBoost + SMOTE"].predict_proba(X_test)[:, 1]
cases = {"XGBoost, t = 0,5": (proba_base, 0.5),
         "XGBoost + SMOTE, t = 0,5": (p_smote, 0.5),
         f"XGBoost, t = {best_t:.2f}".replace(".", ","): (proba_base, best_t)}
cmp = {}
for name, (p, t) in cases.items():
    pred = (p >= t).astype(int)
    cmp[name] = {"F1": f1_score(y_test, pred), "Recall": recall_score(y_test, pred),
                 "Precision": precision_score(y_test, pred),
                 "PR-AUC": average_precision_score(y_test, p)}
print(pd.DataFrame(cmp).T.round(3))


### Ô lệnh 18 -- SMOTE làm "phình" xác suất dự đoán

In [ ]:
print(f"Tỷ lệ vỡ nợ thật trên tập test        : {y_test.mean():.3f}")
print(f"Xác suất vỡ nợ TB -- XGBoost          : {proba_base.mean():.3f}")
print(f"Xác suất vỡ nợ TB -- XGBoost + SMOTE  : {p_smote.mean():.3f}")


### Ô lệnh 19 -- SMOTE "phá hỏng" các biến phân loại

In [ ]:
X_res_raw, _ = SMOTE(random_state=SEED).fit_resample(X_train, y_train)  # dữ liệu gốc cho dễ đọc
syn_raw = X_res_raw.iloc[len(X_train):]
print("Tỷ lệ mẫu tổng hợp có EDUCATION không phải số nguyên:",
      f"{(syn_raw['EDUCATION'] % 1 != 0).mean():.1%}")
print(syn_raw[["SEX", "EDUCATION", "MARRIAGE", "PAY_1"]].head(5).round(2))


### Ô lệnh 20 -- SMOTE-NC: phiên bản dành cho dữ liệu có biến phân loại

In [ ]:
from imblearn.over_sampling import SMOTENC

cat_cols = ["SEX", "EDUCATION", "MARRIAGE"]
smnc = SMOTENC(categorical_features=[X_train.columns.get_loc(c) for c in cat_cols],
               random_state=SEED)
X_nc, y_nc = smnc.fit_resample(X_train, y_train)
syn_nc = X_nc.iloc[len(X_train):]
print("Tỷ lệ mẫu tổng hợp có EDUCATION không phải số nguyên:",
      f"{(syn_nc['EDUCATION'] % 1 != 0).mean():.1%}")
print(syn_nc[cat_cols + ["PAY_1"]].head(5).round(2))
